# BitirmeProject — gemma3:4b QLoRA Fine-tune (v1)

**Faz 2 uygulaması.** Colab Free T4 (16 GB VRAM) için yazıldı.
Dataset lokalden hazırlanıp Drive'a yüklenir; eğitim burada koşar; adapter + GGUF Drive'a yazılır.

**Plan:** `docs/ai-agent-fine-tuning-plan.md` Faz 2.  
**Hedef davranışlar:** `docs/ai-agent-target-behaviors.md`.  
**Hyperparametreler:** `tools/ai-finetune/configs/v1.yaml`.  
**Eval seti:** `tests/AiEvalDataset/v1/*.jsonl` (eğitime DAHİL DEĞİL).

## Akış
1. GPU kontrolü
2. Bağımlılıklar (Unsloth + peft + trl + bitsandbytes)
3. Drive mount + dataset path + config yükle
4. Base model + LoRA adapter
5. Dataset yükle + chat template format
6. SFTTrainer — eğit (checkpoint Drive'a)
7. Quick eval (5 örnek × 3 feature)
8. Export — LoRA adapter + GGUF q4_k_m

**Colab disconnect:** `resume_from_checkpoint=True` ile son checkpoint'ten devam eder.

## 1) GPU kontrolü

Colab Runtime → Change runtime type → T4 GPU seçili olmalı.

In [ ]:
!nvidia-smi

## 2) Bağımlılıklar

Unsloth resmi kurulum sırası — bozmak bitsandbytes CUDA uyumsuzluğu yaratıyor.

In [ ]:
%%capture
!pip install --upgrade --force-reinstall pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets pyyaml

## 3) Drive mount + config yükle

Beklenen Drive düzeni:
```
MyDrive/bp-finetune/
  datasets/
    train-v1.train.jsonl   # prepare_dataset.py çıktısı
    train-v1.val.jsonl
  configs/
    v1.yaml                # tools/ai-finetune/configs/v1.yaml kopyası
  adapters/                # çıkış
  checkpoints/             # her 100 step'te flush
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, json
BASE = '/content/drive/MyDrive/bp-finetune'
TRAIN_PATH = f'{BASE}/datasets/train-v1.train.jsonl'
VAL_PATH = f'{BASE}/datasets/train-v1.val.jsonl'
CONFIG_PATH = f'{BASE}/configs/v1.yaml'
ADAPTER_OUT = f'{BASE}/adapters/gemma3-4b-bp-v1'
CKPT_DIR = f'{BASE}/checkpoints'

for d in (ADAPTER_OUT, CKPT_DIR, f'{BASE}/adapters'):
    os.makedirs(d, exist_ok=True)

for p in (TRAIN_PATH, VAL_PATH, CONFIG_PATH):
    assert os.path.exists(p), f'eksik: {p}. Lokalden upload et.'

with open(CONFIG_PATH, encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

print('config:', CFG['version'], '| base:', CFG['base_model']['name'])
print('lr:', CFG['training']['learning_rate'], '| epochs:', CFG['training']['num_train_epochs'])

## 4) Base model + LoRA adapter

`unsloth/gemma-3-4b-it-bnb-4bit` — 4-bit quantized, T4'te ~5 GB VRAM ile yüklenir.

In [ ]:
from unsloth import FastLanguageModel
import torch

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

bm = CFG['base_model']
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=bm['name'],
    max_seq_length=bm['max_seq_length'],
    dtype=None,
    load_in_4bit=bm['load_in_4bit'],
)

lc = CFG['lora']
model = FastLanguageModel.get_peft_model(
    model,
    r=lc['r'],
    lora_alpha=lc['alpha'],
    lora_dropout=lc['dropout'],
    bias=lc['bias'],
    target_modules=lc['target_modules'],
    use_gradient_checkpointing=lc['use_gradient_checkpointing'],
    random_state=lc['random_state'],
)

model.print_trainable_parameters()

## 5) Dataset yükle + chat template

`messages=[system,user,assistant]` formatı — `tokenizer.apply_chat_template` gemma chat template'ini uygular.

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={'train': TRAIN_PATH, 'validation': VAL_PATH})
print(ds)

def _format(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

ds = ds.map(_format, remove_columns=[c for c in ds['train'].column_names if c != 'messages'])
print('örnek:')
print(ds['train'][0]['text'][:600], '...')

## 6) SFTTrainer — eğit

Checkpoint her 100 step'te Drive'a flush — Colab disconnect olursa `resume_from_checkpoint=True`.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

tc = CFG['training']
args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=tc['per_device_train_batch_size'],
    gradient_accumulation_steps=tc['gradient_accumulation_steps'],
    learning_rate=tc['learning_rate'],
    lr_scheduler_type=tc['lr_scheduler_type'],
    warmup_ratio=tc['warmup_ratio'],
    weight_decay=tc['weight_decay'],
    optim=tc['optim'],
    fp16=tc['fp16'],
    bf16=tc['bf16'],
    logging_steps=tc['logging_steps'],
    save_steps=tc['save_steps'],
    save_total_limit=tc['save_total_limit'],
    eval_steps=tc['eval_steps'],
    evaluation_strategy=tc['evaluation_strategy'],
    report_to=tc['report_to'],
    seed=tc['seed'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    dataset_text_field='text',
    max_seq_length=CFG['base_model']['max_seq_length'],
    packing=CFG['dataset']['packing'],
    args=args,
)

last_ckpt = None
if os.path.isdir(CKPT_DIR):
    ckpts = sorted([d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')],
                   key=lambda x: int(x.split('-')[1]))
    if ckpts:
        last_ckpt = os.path.join(CKPT_DIR, ckpts[-1])
        print(f'resume from {last_ckpt}')

trainer.train(resume_from_checkpoint=last_ckpt)

## 7) Quick eval — 5 örnek × 3 feature

Gerçek eval Faz 3'te `tools/ai-eval/runner.py` ile yapılacak. Burada sadece sanity check.

In [ ]:
import json
from pathlib import Path

FastLanguageModel.for_inference(model)

SYSTEM = {
    'scaffold-project': "Sen BitirmeProject AI agent'ısın. Kullanıcının serbest metin talebinden ProjectDraft JSON'u üretirsin. Yalnızca geçerli JSON döndürürsün; markdown fence, açıklama, İngilizce sızıntısı yasak. Türkçe yanıt ver.",
    'enrich-issue':     "Sen BitirmeProject AI agent'ısın. Verilen issue başlığından detaylı issue spesifikasyonu üretirsin (description, acceptanceCriteria, edgeCases, storyPoints). Yalnızca geçerli JSON döndürürsün. Türkçe.",
    'generate-plan':    "Sen BitirmeProject AI agent'ısın. Mevcut projeye yeni özellik sprint planı üretirsin. Yalnızca geçerli JSON (sprints[]) döndürürsün. Türkçe.",
}

EVAL_DIR = f'{BASE}/eval/v1'  # lokal repodan Drive'a kopyalayarak oluştur
N = CFG['quick_eval']['sample_size']

summary = {'total': 0, 'parse_ok': 0, 'per_feature': {}}

for feat in ('scaffold-project', 'enrich-issue', 'generate-plan'):
    path = Path(EVAL_DIR) / f'{feat}.jsonl'
    if not path.exists():
        print(f'[skip] {path}')
        continue
    samples = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
            if len(samples) >= N:
                break

    ok = 0
    for ex in samples:
        msgs = [
            {'role': 'system', 'content': SYSTEM[feat]},
            {'role': 'user', 'content': json.dumps(ex['input'], ensure_ascii=False)},
        ]
        inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
        gen = model.generate(
            inputs,
            max_new_tokens=CFG['quick_eval']['max_new_tokens'],
            temperature=CFG['quick_eval']['temperature'],
            top_p=CFG['quick_eval']['top_p'],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(gen[0][inputs.shape[1]:], skip_special_tokens=True)
        try:
            start = text.find('{'); end = text.rfind('}')
            json.loads(text[start:end + 1])
            ok += 1
        except Exception:
            pass

    summary['total'] += len(samples)
    summary['parse_ok'] += ok
    summary['per_feature'][feat] = f'{ok}/{len(samples)}'
    print(f'[{feat}] parse {ok}/{len(samples)}')

print('---')
print(f"Format compliance: {summary['parse_ok']}/{summary['total']}")
print(summary['per_feature'])

## 8) Export — LoRA adapter + GGUF

Adapter klasörü Drive'da kalır; GGUF Ollama `Modelfile`'da `ADAPTER` olarak gösterilir.

In [ ]:
# LoRA adapter
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'[adapter] {ADAPTER_OUT}')

# GGUF (Unsloth helper — llama.cpp convert kullanır, Colab'da bundled)
gguf_quant = CFG['export']['gguf_quantization']
model.save_pretrained_gguf(ADAPTER_OUT, tokenizer, quantization_method=gguf_quant)

# Çıktıları listele
for f in sorted(os.listdir(ADAPTER_OUT)):
    p = os.path.join(ADAPTER_OUT, f)
    sz = os.path.getsize(p) / 1024 / 1024 if os.path.isfile(p) else 0
    print(f'{f}: {sz:.1f} MB' if sz else f'{f}/')

# Ollama Modelfile taslağı
modelfile = f"""FROM gemma3:4b
ADAPTER ./gemma3-4b-bp-v1-{gguf_quant}.gguf
PARAMETER temperature 0.3
PARAMETER top_p 0.9
SYSTEM \"Sen BitirmeProject AI agent'ısın. Yalnızca geçerli JSON döndürürsün.\"
"""
with open(f'{ADAPTER_OUT}/Modelfile', 'w', encoding='utf-8') as f:
    f.write(modelfile)
print('Modelfile yazıldı.')